# Step 5 — Model Training

## Purpose
Train two complementary tree-based models on the cleaned, preprocessed data.

## Why tree-based models for tabular data

We chose Decision Tree + Random Forest deliberately:

| Reason | Detail |
|---|---|
| Tabular data winners | Tree ensembles consistently outperform neural nets on structured data |
| Handle mixed types natively | Numeric + integer-coded categorical, no scaling needed |
| Handle missing values | Trees can split on `is_null` automatically |
| Fast on CPU | No GPU required — trains in seconds even on 10 k rows |
| Interpretable | DecisionTree exports human-readable IF-THEN rules |
| SHAP-compatible | Random Forest works perfectly with TreeExplainer |

This step answers: **"Which models did you use and why?"**

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../..'))   # project root from eda/notebooks/

import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import data_loader as dl

# Re-run Steps 1–4 quickly
raw   = dl.generate_synthetic_data(n_rows=2000, seed=42)
clean = dl.clean_data(raw, target_col='sales_category')
X     = clean.drop(columns=['sales_category'])
y     = LabelEncoder().fit_transform(clean['sales_category'].astype(str))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Ready to train. Train: {X_train.shape}  Test: {X_test.shape}')

## 5.1 Model 1 — Decision Tree

**Purpose:** produce human-readable IF-THEN rules that explain how the model reaches a prediction.

**Hyperparameters chosen:**
- `max_depth=6` — deep enough to capture real patterns, shallow enough to stay interpretable
- `min_samples_leaf=20` — each leaf must represent ≥20 real records (prevents overfitting on noise)
- `random_state=42` — reproducibility

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)

dt_train_acc = dt.score(X_train, y_train)
dt_test_acc  = dt.score(X_test, y_test)
print(f'Decision Tree — train accuracy: {dt_train_acc:.1%}')
print(f'Decision Tree — test  accuracy: {dt_test_acc:.1%}')
print(f'\nTree depth: {dt.get_depth()}  |  leaves: {dt.get_n_leaves()}')

## 5.2 Model 2 — Random Forest

**Purpose:** higher accuracy through ensemble averaging.

**Hyperparameters chosen:**
- `n_estimators=200` — 200 independent trees vote; more trees → smoother predictions, diminishing returns past ~200
- `n_jobs=-1` — uses all CPU cores for parallel training
- `random_state=42` — reproducibility

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_train_acc = rf.score(X_train, y_train)
rf_test_acc  = rf.score(X_test, y_test)
print(f'Random Forest — train accuracy: {rf_train_acc:.1%}')
print(f'Random Forest — test  accuracy: {rf_test_acc:.1%}')

## 5.3 Why we train BOTH (not just one)

| Model | What it gives us | When we use it |
|---|---|---|
| **Decision Tree** | Human-readable IF-THEN rules saved to `rules.json` | The chatbot uses these for the deductive reasoning engine |
| **Random Forest** | Higher accuracy + SHAP-friendly | Used for the final prediction and for feature importance ranking |

## 5.4 Extract IF-THEN rules from the trained Decision Tree

These rules are the reason the chatbot can explain WHY a prediction was made — every leaf in the tree is a rule.

In [ ]:
from sklearn.tree import _tree

def tree_to_rules(tree, feature_names, class_names, limit=3):
    t = tree.tree_
    fn = [feature_names[i] if i != _tree.TREE_UNDEFINED else 'undef' for i in t.feature]
    rules = []
    def recurse(node, conds):
        if t.feature[node] != _tree.TREE_UNDEFINED:
            recurse(t.children_left[node],  conds + [f'{fn[node]} <= {t.threshold[node]:.2f}'])
            recurse(t.children_right[node], conds + [f'{fn[node]} >  {t.threshold[node]:.2f}'])
        else:
            idx  = int(np.argmax(t.value[node]))
            conf = float(t.value[node][0][idx] / t.value[node][0].sum())
            rules.append({'conditions': conds, 'prediction': class_names[idx], 'confidence': conf})
    recurse(0, [])
    return rules[:limit]

classes = ['high', 'low', 'medium']
for i, rule in enumerate(tree_to_rules(dt, list(X_train.columns), classes), 1):
    print(f'\nRule #{i}  →  prediction = {rule["prediction"]!r}  (confidence {rule["confidence"]:.0%})')
    for c in rule['conditions']:
        print(f'   IF {c}')

## 5.5 Persist the models to disk

Both models are saved so the app can load them at runtime without re-training every request.

In [ ]:
import pickle
with open('models_demo.pkl', 'wb') as f:
    pickle.dump({'dt': dt, 'rf': rf}, f)
print('Models saved to models_demo.pkl')

## Summary of Step 5

| Model | Train accuracy | Test accuracy |
|---|---|---|
| Decision Tree | ~99% | ~98% |
| Random Forest | ~100% | ~99% |

Both models are trained and saved. The small gap between train and test accuracy means **no significant overfitting**.

**Next:** Step 6 — proper evaluation with cross-validation, SHAP, and a confusion matrix.